In [4]:
import pandas as pd 


In [5]:
datar = pd.read_csv('../data/processed/paes_valdivia.csv')
datar.head(15)

# Limpiar RBD: quedarse solo con lo anterior al "-"
datar["rbd"] = (
    datar["rbd"]
    .astype(str)
    .str.split("-")
    .str[0]
)

# Guardar archivo limpio
datar.to_csv("../data/processed/paes_valdivia_limpio.csv", index=False)
datar



,anio,establecimiento,rbd,prueba,promedio
0,2024,Escuela Particular El Prado\nRama,22158,Lenguaje,473.0
1,2024,Escuela Particular El Prado\nRama,22158,M1,392.0
2,2024,Escuela Particular El Prado\nRama,22158,M2,348.0
3,2024,Escuela Particular El Prado\nRama,22158,Historia,0.0
4,2024,Escuela Particular El Prado\nRama,22158,Ciencias,425.0
...,...,...,...,...,...
931,2026,Liceo Antonio Varas\nRama,7299,M1,614.6
932,2026,Liceo Antonio Varas\nRama,7299,M2,401.5
933,2026,Liceo Antonio Varas\nRama,7299,Historia,505.1
934,2026,Liceo Antonio Varas\nRama,7299,Ciencias,420.4


In [6]:


datac = pd.read_csv('../data/colegios/20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv', sep=';')
datac
datac_subset = datac[["RBD", "DGV_RBD", "NOM_RBD", "NOM_COM_RBD", "LATITUD", "LONGITUD"]]
datac_subset.head(12)



FileNotFoundError: [Errno 2] No such file or directory: '../data/colegios/20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv'

In [ ]:
# Filtrar filas por valor exacto
valdivia = datac_subset[datac_subset["NOM_COM_RBD"] == "VALDIVIA"]
valdivia


,RBD,DGV_RBD,NOM_RBD,NOM_COM_RBD,LATITUD,LONGITUD
5560,6751,2,INSTITUTO COMERCIAL DE VALDIVIA,VALDIVIA,"-39,817619","-73,246964"
5561,6752,0,LICEO TECNICO VALDIVIA,VALDIVIA,"-39,82925","-73,221237"
5562,6753,9,LICEO SANTA MARIA LA BLANCA,VALDIVIA,"-39,81601","-73,240768"
5563,6754,7,LICEO ARMANDO ROBLES RIVERA,VALDIVIA,"-39,81567","-73,242622"
5564,6755,5,LICEO INDUSTRIAL VALDIVIA,VALDIVIA,"-39,831871","-73,220757"
...,...,...,...,...,...,...
16566,42067,0,INDEPENDENCIA,VALDIVIA,,
16568,42072,7,JARDIN INFANTIL ESTRELLITA NUESTRA,VALDIVIA,,
16647,42202,9,ALTO GUACAMAYO,VALDIVIA,,
16667,42248,7,SALA CUNA Y JARDIN INFANTIL MIS HUELLAS,VALDIVIA,,


In [ ]:
paes = pd.read_csv("../data/processed/paes_valdivia_limpio.csv")
mineduc = pd.read_csv("../data/colegios/establecimientos_mineduc.csv", sep=';')

print(paes["rbd"].head())
print(mineduc["RBD"].head())

print(paes["rbd"].dtype, mineduc["RBD"].dtype)


0    6757
1    6757
2    6757
3    6757
4    6757
Name: rbd, dtype: int64
0    1
1    2
2    3
3    4
4    5
Name: RBD, dtype: int64
int64 int64


In [ ]:
paes["rbd"] = paes["rbd"].astype(str).str.strip()
mineduc["rbd"] = mineduc["RBD"].astype(str).str.strip()


In [ ]:
paes_geo = paes.merge(
    mineduc[["rbd", "LATITUD", "LONGITUD"]],
    on="rbd",
    how="left"
)

sin_coord = paes_geo["LATITUD"].isna().sum()

print(f"Colegios sin coordenadas: {sin_coord}")


paes_geo.to_csv(
    "../data/processed/paes_valdivia_geo.csv",
    index=False
)



Colegios sin coordenadas: 0


In [ ]:
paes_geo

,anio,establecimiento,rbd,prueba,promedio,LATITUD,LONGITUD
0,2024,Liceo Politecnico Benjamin Vicuna Mackenna\nRama,6757,Lenguaje,470.5,"-39,826889","-73,206703"
1,2024,Liceo Politecnico Benjamin Vicuna Mackenna\nRama,6757,M1,457.0,"-39,826889","-73,206703"
2,2024,Liceo Politecnico Benjamin Vicuna Mackenna\nRama,6757,M2,290.0,"-39,826889","-73,206703"
3,2024,Liceo Politecnico Benjamin Vicuna Mackenna\nRama,6757,Historia,396.0,"-39,826889","-73,206703"
4,2024,Liceo Politecnico Benjamin Vicuna Mackenna\nRama,6757,Ciencias,0.0,"-39,826889","-73,206703"
...,...,...,...,...,...,...,...
313,2026,Instituto Aleman Carlos Anwandter\nRama,6844,M1,787.1,"-39,811772","-73,25277"
314,2026,Instituto Aleman Carlos Anwandter\nRama,6844,M2,509.9,"-39,811772","-73,25277"
315,2026,Instituto Aleman Carlos Anwandter\nRama,6844,Historia,600.4,"-39,811772","-73,25277"
316,2026,Instituto Aleman Carlos Anwandter\nRama,6844,Ciencias,585.1,"-39,811772","-73,25277"


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

# =========================
# RUTAS
# =========================
INPUT = Path("../data/processed/paes_valdivia_geo.csv")
OUTPUT = Path("../data/processed/paes_valdivia.geojson")



# =========================
# CARGA
# =========================
df = pd.read_csv(INPUT)

# =========================
# LIMPIEZA COORDENADAS
# =========================
df["LATITUD"] = (
    df["LATITUD"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df["LONGITUD"] = (
    df["LONGITUD"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)


# Seguridad básica
df = df.dropna(subset=["LATITUD", "LONGITUD"])

# =========================
# CREAR GEOMETRÍA
# =========================
geometry = [
    Point(xy) for xy in zip(df["LONGITUD"], df["LATITUD"])
]

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

# =========================
# EXPORTAR A GEOJSON
# =========================
gdf.to_file(
    OUTPUT,
    driver="GeoJSON",
    encoding="utf-8"
)

print("✅ GeoJSON creado correctamente")
print(f"📍 Archivo: {OUTPUT}")
print(f"🔢 Registros: {len(gdf)}")


✅ GeoJSON creado correctamente
📍 Archivo: ..\data\processed\paes_valdivia.geojson
🔢 Registros: 318


In [ ]:
gdf["rbd"] = (
    gdf["rbd"]
    .astype(str)
    .str.split("-")
    .str[0]
)


In [ ]:
gdf_pivot = gdf.pivot_table(
    index=["rbd", "establecimiento", "geometry"],
    columns="prueba",
    values="promedio",
    aggfunc="mean"
).reset_index()
gdf_pivot


prueba,rbd,establecimiento,geometry,Ciencias,Historia,Lenguaje,M1,M2,Obligatoria
0,12185,Colegio Bicentenario Helvecia\nRama,POINT (-73.21696 -39.84378),423.300000,474.550000,551.000000,558.600000,391.100000,555.350000
1,22014,Colegio Baquedano\nRama,POINT (-73.23893 -39.82272),469.500000,516.600000,619.200000,632.700000,423.500000,626.900000
2,22160,Martin Luther King College\nRama,POINT (-73.24035 -39.83623),469.500000,516.600000,619.200000,632.700000,423.500000,626.900000
3,22177,Colegio San Luis De Alba\nRama,POINT (-73.28399 -39.84522),585.100000,600.400000,713.800000,787.100000,509.900000,750.400000
4,22188,Centro Educacion Adultos Andres Bello\nRama,POINT (-73.23983 -39.81742),377.100000,432.500000,482.800000,484.500000,358.700000,483.800000
5,22231,Colegio Domus Mater\nRama,POINT (-73.24646 -39.82877),585.100000,600.400000,713.800000,787.100000,509.900000,750.400000
6,22351,Instituto Inmaculada Concepcion\nRama,POINT (-73.24644 -39.81818),469.500000,516.600000,619.200000,632.700000,423.500000,626.900000
7,22374,Instituto Tecnologico Del Sur\nRama,POINT (-73.21410 -39.82331),423.300000,474.550000,551.000000,558.600000,391.100000,555.350000
8,22388,Centro De Educ,POINT (-73.24007 -39.81233),377.100000,432.500000,482.800000,484.500000,358.700000,483.800000
9,22398,Colegio Aliwen\nRama,POINT (-73.25021 -39.82786),469.500000,516.600000,619.200000,632.700000,423.500000,626.900000


In [ ]:
import geopandas as gpd

gdf_colegios = gpd.GeoDataFrame(
    gdf_pivot,
    geometry="geometry",
    crs="EPSG:4326"
)


In [ ]:
print("Colegios únicos:", gdf_colegios["rbd"].nunique())
print("Filas totales:", len(gdf_colegios))


Colegios únicos: 43
Filas totales: 44


In [ ]:
gdf_colegios.to_file(
    "../data/processed/colegios_paes_valdivia.geojson",
    driver="GeoJSON"
)

print("✅ GeoJSON consolidado por RBD creado correctamente")


✅ GeoJSON consolidado por RBD creado correctamente
